# 项目一：模型比较——逻辑回归、随机森林与 XGBoost

目标：在相同的开发数据、相同的预处理规则和相同的 5 折交叉验证下，公平比较三个候选模型。

最终测试集继续封存。本节的交叉验证只在开发集上进行，用于选择下一步重点优化的候选模型。

## 为什么先看 ROC-AUC，再讨论阈值

本项目之后会根据漏报成本调整阈值。ROC-AUC 衡量模型把风险学生排在前面的能力，不依赖某一个固定阈值，因此适合先比较模型本身的排序能力。

同时也会报告默认阈值 0.5 下的 Recall、Precision 与 F1，作为当前默认预警规则的参考。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [2]:
data_path = Path('projects/01_student_risk_prediction/data/raw/student-mat.csv')
if not data_path.exists():
    data_path = Path.home() / 'solo_work/算法工程师/projects/01_student_risk_prediction/data/raw/student-mat.csv'

df = pd.read_csv(data_path, sep=';')
y = (df['G3'] < 10).astype(int)
X = df.drop(columns=['G1', 'G2', 'G3'])

# 只分出最终测试集；下方的模型选择全部在 X_develop 中用交叉验证完成。
X_develop, X_test, y_develop, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X_develop.select_dtypes(include='number').columns.tolist()
categorical_features = X_develop.select_dtypes(exclude='number').columns.tolist()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
def make_preprocessor():
    # 每个 Pipeline 都得到一份独立预处理器；每一折只在该折训练部分学习统计量和类别字典。
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])

models = {
    'logistic regression': Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'random forest': Pipeline([
        ('preprocessor', make_preprocessor()),
        # 多棵树投票；n_jobs=1 避免与交叉验证的并行任务嵌套。
        ('classifier', RandomForestClassifier(
            n_estimators=300, random_state=42, n_jobs=1
        )),
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', make_preprocessor()),
        # 保守的起始参数；尚未用验证结果对它做针对性调参。
        ('classifier', XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=3,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            eval_metric='logloss', random_state=42, n_jobs=1
        )),
    ]),
}

In [4]:
scoring = {
    'accuracy': 'accuracy',
    'precision_at_risk': 'precision',
    'recall_at_risk': 'recall',
    'f1_at_risk': 'f1',
    'roc_auc': 'roc_auc',
}

rows = []
for name, model in models.items():
    # 每个模型做 5 次临时训练；每次 4 折训练、1 折验证。
    scores = cross_validate(
        model, X_develop, y_develop, cv=cv, scoring=scoring, n_jobs=-1
    )
    rows.append({
        'model': name,
        'accuracy_mean': scores['test_accuracy'].mean(),
        'recall_mean': scores['test_recall_at_risk'].mean(),
        'precision_mean': scores['test_precision_at_risk'].mean(),
        'f1_mean': scores['test_f1_at_risk'].mean(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
        'roc_auc_std': scores['test_roc_auc'].std(),
    })

comparison = pd.DataFrame(rows).sort_values('roc_auc_mean', ascending=False)
display(comparison.round(3))

,model,accuracy_mean,recall_mean,precision_mean,f1_mean,roc_auc_mean,roc_auc_std
1,random forest,0.680,0.232,0.668,0.322,0.664,0.050
2,XGBoost,0.674,0.367,0.529,0.420,0.658,0.067
0,logistic regression,0.674,0.338,0.548,0.401,0.633,0.047


## 如何解读这张表

1. 先比较 `roc_auc_mean`，评估哪种模型的风险排序更有效。
2. 观察 `roc_auc_std`，波动小意味着模型在不同样本划分下更稳定。
3. 再结合默认阈值下的 Recall 与 Precision，判断当前预警规则会造成怎样的漏报/误报。
4. 分数接近时，不应因模型更复杂就自动选择它；要考虑可解释性、调参成本和项目数据量。

本项目数据只有 316 条开发记录，因此本节结果是模型比较的依据，而不是对真实世界效果的保证。下一步会为表现最好的候选模型单独选择适合预警业务的阈值。